# Parking Birmingham — Feature Engineering and Temporal Splitting

**Course:** ADSP 32021 — Machine Learning Operations  
**Project:** Parking Availability Prediction Using an Automated MLOps Pipeline

## 1. Introduction

This notebook develops leakage-safe features for predicting parking occupancy approximately 30 minutes into the future using the validated UCI Parking Birmingham dataset.

### Objectives

1. Load and validate parking occupancy observations.
2. Construct future occupancy targets using a 25–35-minute forecasting window.
3. Engineer temporal, facility-specific, and historical occupancy features.
4. Prevent data leakage by ensuring features use only information available at prediction time.
5. Split observations chronologically into training, validation, and test sets.
6. Prepare reproducible datasets for baseline model development.

### Data Leakage Prevention

All input features must be available at or before the prediction timestamp.

Future occupancy values will be used exclusively as prediction targets, never as input features.

Chronological splitting will preserve the temporal ordering of observations, and examples whose target timestamps cross split boundaries will be excluded.


## 2. Data Loading and Validation

The raw Parking Birmingham dataset is loaded and processed using the reusable `validate_parking_data()` function developed during the initial data audit.

This function removes duplicate observations, excludes invalid occupancy measurements, converts timestamps to datetime format, and sorts observations chronologically within each parking facility.

Reusing the validation module ensures consistent preprocessing across exploratory analysis, model training, and the future automated Airflow pipeline.

The original dataset remains unchanged to preserve reproducibility.


In [1]:
import pandas as pd
import sys
from pathlib import Path

# Locate the project root from the notebooks directory
project_root = Path.cwd().resolve().parent

# Allow imports from the project's src directory
sys.path.insert(0, str(project_root))

from src.data_validation import validate_parking_data

# Load the original dataset
data_path = project_root / "data" / "dataset.csv"
df_raw = pd.read_csv(data_path)

# Apply the reusable validation function
df = validate_parking_data(df_raw)

# Verify the validated dataset
print("Original observations:", len(df_raw))
print("Validated observations:", len(df))
print("Parking facilities:", df["SystemCodeNumber"].nunique())

print("\nData types:")
print(df.dtypes)

display(df.head())


Original observations: 35717
Validated observations: 35116
Parking facilities: 30

Data types:
SystemCodeNumber            object
Capacity                     int64
Occupancy                    int64
LastUpdated         datetime64[ns]
dtype: object


,SystemCodeNumber,Capacity,Occupancy,LastUpdated
0,BHMBCCMKT01,577,61,2016-10-04 07:59:42
1,BHMBCCMKT01,577,64,2016-10-04 08:25:42
2,BHMBCCMKT01,577,80,2016-10-04 08:59:42
3,BHMBCCMKT01,577,107,2016-10-04 09:32:46
4,BHMBCCMKT01,577,150,2016-10-04 09:59:48



### 2.1 Validated Dataset Summary

The reusable validation pipeline successfully processed the original dataset.

- Original observations: 35,717
- Validated observations: 35,116
- Parking facilities retained: 30
- Timestamp format: datetime64[ns]

The validated dataset is sorted chronologically within each parking facility and contains no duplicate facility-timestamp records or invalid occupancy measurements.

## 3. Forecast Target Construction

The forecasting objective is to predict parking occupancy approximately 30 minutes after the current observation.

Because timestamps are irregular, we define an acceptable prediction horizon of **25–35 minutes**.

For each parking facility, the next recorded occupancy is selected as the target only when its timestamp falls within this interval.

Observations without an eligible future measurement are excluded from the supervised modeling dataset.

**Leakage prevention:** Future occupancy and target timestamps are used only for target construction and evaluation. They will not be included as model input features.

In [2]:
# Ensure observations are sorted within each facility
df = df.sort_values(
    ["SystemCodeNumber", "LastUpdated"]
).reset_index(drop=True)

# Identify the next observation within each facility
facility_groups = df.groupby("SystemCodeNumber")

df["TargetTimestamp"] = (
    facility_groups["LastUpdated"].shift(-1)
)

df["TargetOccupancy"] = (
    facility_groups["Occupancy"].shift(-1)
)

# Calculate the actual forecasting horizon
df["ForecastHorizonMinutes"] = (
    (df["TargetTimestamp"] - df["LastUpdated"])
    .dt.total_seconds() / 60
)

# Keep only observations with a 25–35 minute horizon
df_model = df.loc[
    df["ForecastHorizonMinutes"].between(25, 35)
].copy()

# Verify target construction
print("Validated observations:", len(df))
print("Valid forecasting examples:", len(df_model))

print("\nForecast horizon statistics (minutes):")
print(df_model["ForecastHorizonMinutes"].describe())

print("\nMissing target values:",
      df_model["TargetOccupancy"].isna().sum())

display(
    df_model[
        [
            "SystemCodeNumber",
            "LastUpdated",
            "Occupancy",
            "TargetTimestamp",
            "TargetOccupancy",
            "ForecastHorizonMinutes"
        ]
    ].head(10)
)


Validated observations: 35116
Valid forecasting examples: 31327

Forecast horizon statistics (minutes):
count    31327.000000
mean        29.988520
std          3.318462
min         25.916667
25%         26.966667
50%         29.983333
75%         33.033333
max         34.433333
Name: ForecastHorizonMinutes, dtype: float64

Missing target values: 0


,SystemCodeNumber,LastUpdated,Occupancy,TargetTimestamp,TargetOccupancy,ForecastHorizonMinutes
0,BHMBCCMKT01,2016-10-04 07:59:42,61,2016-10-04 08:25:42,64.0,26.000000
1,BHMBCCMKT01,2016-10-04 08:25:42,64,2016-10-04 08:59:42,80.0,34.000000
2,BHMBCCMKT01,2016-10-04 08:59:42,80,2016-10-04 09:32:46,107.0,33.066667
3,BHMBCCMKT01,2016-10-04 09:32:46,107,2016-10-04 09:59:48,150.0,27.033333
4,BHMBCCMKT01,2016-10-04 09:59:48,150,2016-10-04 10:26:49,177.0,27.016667
5,BHMBCCMKT01,2016-10-04 10:26:49,177,2016-10-04 10:59:48,219.0,32.983333
6,BHMBCCMKT01,2016-10-04 10:59:48,219,2016-10-04 11:25:47,247.0,25.983333
7,BHMBCCMKT01,2016-10-04 11:25:47,247,2016-10-04 11:59:44,259.0,33.950000
8,BHMBCCMKT01,2016-10-04 11:59:44,259,2016-10-04 12:29:45,266.0,30.016667
9,BHMBCCMKT01,2016-10-04 12:29:45,266,2016-10-04 13:02:48,269.0,33.050000
